In [2]:
import os
import pandas as pd
from PIL import Image
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import letter, landscape
from reportlab.lib.utils import ImageReader
import io

In [18]:
cta_results_file = pd.read_csv('/Users/rushil/ichseg/local_results/Rushil_QC_CTA_results.csv')
cta_df = pd.DataFrame(cta_results_file)
#find the index of the highest total_count in the column
highest_index = cta_df['Total_Count'].idxmax()
highest_dataset = 'synthstrip'  # cta_df.loc[highest_index, 'Method']
print(highest_dataset)

file = pd.read_csv(f'/Users/rushil/ichseg/{highest_dataset}/annotations.csv')
file_df = pd.DataFrame(file)
file_df = file_df[file_df['5 - CTA'] == 'yes']

def generate_pdf(file_df, output_pdf):
    c = canvas.Canvas(output_pdf, pagesize=landscape(letter))
    width, height = landscape(letter)
    for index, row in file_df.iterrows():
        image = row['Filename']
        image_path = os.path.join('/Users/rushil/ichseg', highest_dataset, 'image_ss_' + highest_dataset, image)
        image = Image.open(image_path)
        image_reader = ImageReader(image)

        c.drawImage(image_reader, 0, 0, width=width, height=height)
        c.showPage()

    c.save()
    
output_pdf_path = f'/Users/rushil/ichseg/{highest_dataset}/only_cta_scans.pdf'
generate_pdf(file_df, output_pdf_path)



synthstrip


In [19]:
org_file = pd.read_csv('/Users/rushil/ichseg/synthstrip/annotations.csv')
org_df = pd.DataFrame(org_file)

synth_cta_images = org_df[org_df['5 - CTA'] == 'yes']['Filename'].tolist()
robust_df = pd.DataFrame(pd.read_csv('/Users/rushil/ichseg/ctbet/annotations.csv'))
robust_df_cta_images = robust_df[robust_df['5 - CTA'] == 'yes']['Filename'].tolist()

robust_df_only_cta_images = set(robust_df_cta_images) - set(synth_cta_images)
print("Images in the robust DataFrame that are not in the original DataFrame:")
for image in robust_df_only_cta_images:
    print(image)
    
print(len(robust_df_only_cta_images))


Images in the robust DataFrame that are not in the original DataFrame:
0


In [3]:
methods = ['v1', 'robust', 'hdctbet', 'ctbet', 'brainchop']

org_file = pd.read_csv('/Users/rushil/ichseg/synthstrip/annotations.csv')
org_df = pd.DataFrame(org_file)
org_cta_images = org_df[org_df['5 - CTA'] == 'yes']['Filename'].tolist()

for method in methods:
    original_csv = pd.read_csv(f"/Users/rushil/ichseg/{method}/annotations.csv")
    failures_csv = pd.read_csv(f"/Users/rushil/ichseg/{method}/annotations_failures.csv")
    original_df = pd.DataFrame(original_csv)
    failures_df = pd.DataFrame(failures_csv)
    
    # Set 'yes' for craniotomy image
    original_df.loc[original_df['Filename'].isin(org_cta_images), '5 - CTA'] = 'yes'
    failures_df.loc[failures_df['Filename'].isin(org_cta_images), '5 - CTA'] = 'yes'
    
    # Set 'no' for all other images
    original_df.loc[~original_df['Filename'].isin(org_cta_images), '5 - CTA'] = 'no'
    failures_df.loc[~failures_df['Filename'].isin(org_cta_images), '5 - CTA'] = 'no'
    
    original_df.to_csv(f"/Users/rushil/ichseg/{method}/annotations.csv", index=False)
    failures_df.to_csv(f"/Users/rushil/ichseg/{method}/annotations_failures.csv", index=False)